In [ ]:
# ACTUAL CODE

In [2]:
import numpy as np
import glob as gb
import matplotlib.pyplot as plt
import os
import cv2
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout, Activation, MaxPooling2D, BatchNormalization,AveragePooling2D
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split, KFold
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score, confusion_matrix

In [3]:
# Define dataset path
path = "/content/sample_data/breastData/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT/"

In [4]:
# Function to encode class labels
def encode(f):
    labels = {'malignant': 0, 'benign': 1, 'normal': 2}
    return labels[f]

In [5]:
imageSize = 180

X = []
y = []

In [ ]:
for folder in os.listdir(path):
  for file in gb.glob(path + folder + "/*.png"):
      if "mask" not in file:
        image = cv2.imread(file)
        image = cv2.resize(image, (imageSize, imageSize))
        image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # feature extraction
        image_blur = cv2.GaussianBlur(image_gray, (7, 7), 0)
        _, thresh_image = cv2.threshold(image_blur, 100, 255, cv2.THRESH_BINARY)
        contours, hierarchy = cv2.findContours(thresh_image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        image_contour = cv2.drawContours(image_gray, contours, -1, (0, 255, 0), 2)


        X.append(image_contour)
        y.append(encode(folder))

In [ ]:
X = np.array(X)

# Normalization
X = X/255.0

y = np.array(y)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [ ]:
y_train.shape

(624,)

In [ ]:
CNNmodel = Sequential(
    [
        Conv2D(64, (3, 3), input_shape=(imageSize, imageSize, 1), padding='same', activation='relu'), # First convolutional layer
        MaxPool2D(pool_size=(2, 2)),  # Max pooling to reduce spatial dimensions
        Conv2D(64, (3, 3), padding = 'same', activation='relu'), # Second convolutional layer
        MaxPool2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), padding='same', activation='relu'), # Third convolutional layer
        MaxPool2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), padding='same', activation='relu'), # Fourth convolutional layer
        MaxPool2D(pool_size=(2, 2)),
        Conv2D(256, (3, 3), padding='same', activation='relu'), # Fifth convolutional layer
        MaxPool2D(pool_size=(2, 2)),
        Flatten(), # Flattening layer to convert 2D features into a 1D vector
        Dense(units=512, activation='relu'),
        Dense(units=256, activation='relu'),
        Dense(units=128, activation='relu'),
        Dense(units=3, activation='softmax'),
    ]

)
CNNmodel.summary()
CNNmodel.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = CNNmodel.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 180, 180, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 90, 90, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 22, 22, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 11, 11, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 11, 11, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 5, 5, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     3,277,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,996,099 (15.24 MB)

 Trainable params: 3,996,099 (15.24 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 88s 4s/step - accuracy: 0.5116 - loss: 1.0801 - val_accuracy: 0.5769 - val_loss: 1.0049
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 83s 4s/step - accuracy: 0.5510 - loss: 1.0035 - val_accuracy: 0.5769 - val_loss: 1.0471
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 153s 5s/step - accuracy: 0.5621 - loss: 0.9941 - val_accuracy: 0.5769 - val_loss: 0.9607
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 129s 4s/step - accuracy: 0.5289 - loss: 0.9887 - val_accuracy: 0.5769 - val_loss: 0.9398
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 87s 4s/step - accuracy: 0.4953 - loss: 1.0469 - val_accuracy: 0.5769 - val_loss: 0.9732
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 142s 4s/step - accuracy: 0.5643 - loss: 0.9503 - val_accuracy: 0.5769 - val_loss: 0.9301
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 86s 4s/step - accuracy: 0.5614 - loss: 0.9261 - val_accuracy: 0.6282 - val_loss: 0.9723
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 148s 5s/step - accuracy: 0.6326 - loss: 0.9206 - val_accuracy: 0.6410 - val_l

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Health Disease Risk assessment/breast cancer detection/Models')

In [ ]:
CNNmodel.save('bestBreastprediction.h5')